## Functional API Regression

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Data
X, y=fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


## Model

In [3]:
inputs=keras.Input(shape=(8, ))
shared=keras.layers.Dense(128, activation="relu")(inputs)
shared=keras.layers.Dense(64, activation="relu")(shared)

# Two separate output heads from the same shared backbone
price_out       = keras.layers.Dense(1, name="price")(shared)
confidence_out  = keras.layers.Dense(1, activation="sigmoid", name="confidence")(shared)

model=keras.Model(inputs=inputs, outputs=[price_out, confidence_out])

model.compile(
    optimizer="adam",
    loss={"price":"mse", "confidence": "binary_crossentropy"},
    loss_weights={"price":1.0, "confidence":0.1}
)

confidence_labels=(y_train>np.median(y_train)).astype("float32")


In [4]:
## Train
history=model.fit(
    X_train, 
    {"price": y_train, "confidence":confidence_labels},
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

In [5]:
# confidence labels for test set — use TRAIN median, not test median
conf_test = (y_test > np.median(y_train)).astype("float32")

# ── Evaluate ──────────────────────────────────────────────────────────
results = model.evaluate(
    X_test,
    {"price": y_test, "confidence": conf_test},
    verbose=0
)

# results order: [total_loss, price_loss, confidence_loss]
print(f"Test Total Loss       : {results[0]:.4f}")
print(f"Test Price MSE        : {results[1]:.4f}")
print(f"Test Confidence Loss  : {results[2]:.4f}")

# ── Manual predictions ────────────────────────────────────────────────
price_preds, conf_preds = model.predict(X_test, verbose=0)

print("\nSample predictions:")
for i in range(5):
    above = "above" if conf_preds[i][0] > 0.5 else "below"
    print(
        f"  Predicted: ${price_preds[i][0]:.2f}  |  "
        f"Actual: ${y_test[i]:.2f}  |  "
        f"Confidence: {above} median ({conf_preds[i][0]:.2f})"
    )

Test Total Loss       : 0.2949
Test Price MSE        : 0.2664
Test Confidence Loss  : 0.2848

Sample predictions:
  Predicted: $0.44  |  Actual: $0.48  |  Confidence: below median (0.00)
  Predicted: $1.39  |  Actual: $0.46  |  Confidence: below median (0.13)
  Predicted: $4.90  |  Actual: $5.00  |  Confidence: above median (1.00)
  Predicted: $2.55  |  Actual: $2.19  |  Confidence: above median (0.98)
  Predicted: $2.86  |  Actual: $2.78  |  Confidence: above median (0.98)
